In [1]:
from pathlib import Path

from scipy.optimize import linear_sum_assignment
import torch

In [2]:
x, target = torch.load("batch.pt", weights_only=False)
x.shape, target["boxes"].shape, target["labels"].shape, target["object_mask"].shape

(torch.Size([32, 3, 128, 128]),
 torch.Size([32, 2, 4]),
 torch.Size([32, 2]),
 torch.Size([32, 2]))

In [3]:
target_boxes = target["boxes"]
target_labels = target["labels"]
object_mask = target["object_mask"]

In [4]:
TORCH_SEED = 42
BATCH_SIZE = 32
NUM_CLASSES = 3  # background + rectangle + circle
MAX_NUM_OBJS = 10
IMAGE_SIZE = 128

torch.manual_seed(TORCH_SEED)

pred_locations = torch.rand(BATCH_SIZE, MAX_NUM_OBJS, 4) * IMAGE_SIZE

pred_class_scores = torch.randn(BATCH_SIZE, MAX_NUM_OBJS, NUM_CLASSES)
pred_class_probs = torch.softmax(pred_class_scores, dim=-1)

pred_locations.shape, pred_class_probs.shape

(torch.Size([32, 10, 4]), torch.Size([32, 10, 3]))

# bbox cdist

In [20]:
pred_locations.shape, pred_locations[0]

(torch.Size([32, 10, 4]),
 tensor([[112.9305, 117.1205,  49.0066, 122.7911],
         [ 49.9774,  76.9146,  32.8413, 101.5861],
         [120.4187,  17.0478, 119.6286,  75.9782],
         [111.2838,  72.6676,  94.8600,  54.9638],
         [113.3367,  73.4598,  34.1222,  80.3135],
         [ 34.5129,  56.4945,  38.0059, 106.4557],
         [ 13.4803,  34.4953,  45.9280,  25.5186],
         [ 70.0405,   0.7885, 121.7990,   9.6340],
         [113.4098,  74.6508,  43.2189, 103.5488],
         [ 73.9744, 115.7097,  70.9965,  43.8161]]))

In [21]:
target_boxes.shape, target_boxes[0]

(torch.Size([32, 2, 4]),
 tensor([[ 6., 50., 32., 74.],
         [ 4.,  4., 32., 34.]]))

In [5]:
# Compute pairwise costs against every padded target slot first.
bbox_cost = torch.cdist(pred_locations, target_boxes, p=2)
bbox_cost.shape

torch.Size([32, 10, 2])

# CLASS DISTANCE

In [6]:
pred_class_probs.shape, pred_class_probs[0]

(torch.Size([32, 10, 3]),
 tensor([[0.1671, 0.1696, 0.6633],
         [0.3678, 0.5164, 0.1159],
         [0.6159, 0.2579, 0.1262],
         [0.2270, 0.4838, 0.2891],
         [0.1415, 0.8241, 0.0344],
         [0.2844, 0.3465, 0.3691],
         [0.3836, 0.5248, 0.0915],
         [0.0589, 0.1124, 0.8287],
         [0.5719, 0.2890, 0.1391],
         [0.6762, 0.1609, 0.1629]]))

In [ ]:
target_labels.shape, target_labels[0]

(torch.Size([32, 2]), tensor([2, 2]))

In [ ]:
import torch.nn.functional as F

target_labels_one_hot = F.one_hot(target_labels, num_classes=NUM_CLASSES).float()
target_labels_one_hot.shape, target_labels_one_hot[0]

(torch.Size([32, 2, 3]),
 tensor([[0., 0., 1.],
         [0., 0., 1.]]))

In [19]:
class_cost = torch.cdist(
    pred_class_probs,
    target_labels_one_hot,
    p=2,
)

class_cost.shape

torch.Size([32, 10, 2])

In [13]:
label_indices = target_labels.unsqueeze(1).expand(-1, pred_class_probs.shape[1], -1)
class_cost = -pred_class_probs.gather(dim=2, index=label_indices)
class_cost.shape

torch.Size([32, 10, 2])

In [12]:
pairwise_cost = bbox_cost_weight * bbox_cost + class_cost_weight * class_cost

NameError: name 'bbox_cost_weight' is not defined

In [ ]:


    # Only after the pairwise distances/costs exist, mask padded target slots.
    masked_pairwise_cost = pairwise_cost.masked_fill(
        ~object_mask.unsqueeze(1), torch.inf
    )

    matches = []
    for batch_idx in range(masked_pairwise_cost.shape[0]):
        valid_target_indices = object_mask[batch_idx].nonzero(as_tuple=True)[0]

        if valid_target_indices.numel() == 0:
            matches.append(
                (
                    torch.empty(0, dtype=torch.long),
                    torch.empty(0, dtype=torch.long),
                )
            )
            continue

        sample_cost = masked_pairwise_cost[batch_idx, :, valid_target_indices]
        pred_indices, valid_target_positions = linear_sum_assignment(
            sample_cost.cpu().numpy()
        )

        matches.append(
            (
                torch.as_tensor(pred_indices, dtype=torch.long),
                valid_target_indices[
                    torch.as_tensor(valid_target_positions, dtype=torch.long)
                ].cpu(),
            )
        )

    return matches, pairwise_cost, masked_pairwise_cost


matches, pairwise_cost, masked_pairwise_cost = hungarian_match(
    pred_locations=pred_locations,
    pred_class_probs=pred_class_probs,
    target=target,
)

matches[0], pairwise_cost.shape, masked_pairwise_cost.shape